# Movie Ratings at Scale (MovieLens) - запуск в Colab
Выполняйте ячейки сверху вниз.

## 0. Клонирование репозитория
Замените `GITHUB_URL` на ссылку на ВАШ репозиторий. Для приватного репозитория: `https://<TOKEN>@github.com/<user>/<repo>.git`.

In [ ]:
GITHUB_URL = "https://github.com/YOUR_USER/movielens-ratings.git"  # <- ЗАМЕНИТЬ
REPO_DIR = "movielens-ratings"

import os
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline | head -5

## 1. Установка библиотек

In [ ]:
!pip install -q -r requirements.txt

## 2. Скачивание MovieLens 32M
При нехватке памяти замените `ml-32m` на `ml-25m`.

In [ ]:
DATASET = "ml-32m"
!wget -q -nc https://files.grouplens.org/datasets/movielens/{DATASET}.zip -O /content/{DATASET}.zip
!unzip -q -o /content/{DATASET}.zip -d /content/
!cp /content/{DATASET}/ratings.csv /content/{DATASET}/movies.csv data/raw/
!ls -lh data/raw

## 3. Шаг 1. Аудит качества + память до/после

In [ ]:
import json
from src import pipeline as P
audit, mem = P.step_audit_and_memory()
print(json.dumps({'audit': audit, 'memory': mem}, indent=2, ensure_ascii=False))

## 4. Шаг 2. CSV → Parquet, сравнение

In [ ]:
info, io = P.step_parquet()
print(json.dumps({'convert': info, 'io': io}, indent=2))

## 5. Шаг 3. База данных SQLite (3НФ) + ER-диаграмма
Загрузка 32M оценок занимает несколько минут.

In [ ]:
from IPython.display import Image, display
print(P.step_database())
display(Image('outputs/figures/er_diagram.png'))

## 6. Шаг 4. SQL-запросы Q1-Q10 (результаты в Pandas)

In [ ]:
from src.sql_queries import load_queries, run_query
for qid, (desc, sql) in load_queries().items():
    print('\n', qid, desc)
    display(run_query(qid).head(10))

## 7. Шаг 5. Один расчёт: Pandas(CSV) vs Pandas(Parquet) vs SQL

In [ ]:
print(json.dumps(P.step_compare(), indent=2))

## 8. Шаг 6. EDA и графики

In [ ]:
import glob
summary = P.step_eda()
print(json.dumps(summary, indent=2, ensure_ascii=False))
for f in sorted(glob.glob('outputs/figures/0*.png')):
    display(Image(f))

## 9. Сохранить результаты на Google Drive (по желанию)
Colab стирает файлы после закрытия сессии.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/movielens_results
!cp -r outputs /content/drive/MyDrive/movielens_results/
!cp data/processed/movielens.db /content/drive/MyDrive/movielens_results/